In [ ]:
# Bloco P0 — reproduzir o recorte do PadChest (pacotes 0–11 e 54) a partir do CSV
import ast, os
import numpy as np, pandas as pd, redivis

df = redivis.table("_source_").to_pandas_dataframe()
print("Linhas:", len(df), "| colunas:", len(df.columns))
print(df.columns.tolist())

def lista(s):
    try:
        return [str(x).strip().lower() for x in ast.literal_eval(s)]
    except Exception:
        return []

df["rotulos"] = df["Labels"].apply(lista)
df["pos"] = df["rotulos"].apply(lambda r: "pneumonia" in r)
df["pos_atipica"] = df["rotulos"].apply(lambda r: "pneumonia" in r or "atypical pneumonia" in r)
df["rot_exclude"] = df["rotulos"].apply(lambda r: "exclude" in r)
df["ImageDir"] = pd.to_numeric(df["ImageDir"], errors="coerce")
df["ano_exame"] = pd.to_numeric(df["StudyDate_DICOM"].astype(str).str[:4], errors="coerce")
df["idade"] = df["ano_exame"] - pd.to_numeric(df["PatientBirth"], errors="coerce")

print("\nProjection:", df["Projection"].value_counts(dropna=False).to_dict())
print("MethodLabel:", df["MethodLabel"].value_counts(dropna=False).to_dict())

pacotes = list(range(12)) + [54]
frontais = ["PA", "AP", "AP_horizontal"]

passos = []
x = df;                                         passos.append(("Todas as imagens", x))
x = x[x["ImageDir"].isin(pacotes)];             passos.append(("Pacotes 0–11 e 54", x))
x = x[x["Projection"].isin(frontais)];          passos.append(("Frontais (PA, AP, AP deitado)", x))
x = x[x["idade"].between(18, 100)];             passos.append(("Idade de 18 a 100", x))
x_sem_excl = x[~x["rot_exclude"]];              passos.append(("Sem rótulo 'exclude'", x_sem_excl))

def resumo(nome, t):
    est = t.groupby("StudyID").agg(pos=("pos", "max"), pac=("PatientID", "first"))
    return {"etapa": nome, "imagens": len(t), "estudos": len(est),
            "pacientes": t["PatientID"].nunique(), "positivos_estudo": int(est["pos"].sum())}

print("\n" + pd.DataFrame([resumo(n, t) for n, t in passos]).to_string(index=False))

# uma frontal por estudo: prioridade PA > AP > AP deitado, depois ImageID
prio = {"PA": 0, "AP": 1, "AP_horizontal": 2}
rec = (x_sem_excl.assign(p=x_sem_excl["Projection"].map(prio))
       .sort_values(["StudyID", "p", "ImageID"]).drop_duplicates("StudyID", keep="first"))
print(f"\nEstudos com mais de uma frontal: {(x_sem_excl.groupby('StudyID').size() > 1).sum()}")
print(f"Recorte final: {len(rec)} estudos | {rec['PatientID'].nunique()} pacientes | "
      f"positivos {int(rec['pos'].sum())} | com atípica {int(rec['pos_atipica'].sum())} | "
      f"prevalência {rec['pos'].mean():.1%}")
print("Método do rótulo no recorte:", rec["MethodLabel"].value_counts().to_dict())

print("\nPor pacote (imagens do recorte a ler):")
print(rec.groupby("ImageDir").agg(imagens=("ImageID", "size"), positivos=("pos", "sum")).T.to_string())

cols = ["ImageID", "ImageDir", "StudyID", "PatientID", "idade", "PatientSex_DICOM", "Projection",
        "MethodLabel", "Manufacturer_DICOM", "BitsStored_DICOM", "Rows_DICOM", "Columns_DICOM",
        "WindowCenter_DICOM", "WindowWidth_DICOM", "PhotometricInterpretation_DICOM", "pos", "pos_atipica"]
os.makedirs("/out", exist_ok=True)
rec[[c for c in cols if c in rec.columns]].to_csv("/out/padchest_recorte.csv", index=False)
print("\nSalvo: /out/padchest_recorte.csv")

In [ ]:
# Bloco P1 — acesso e velocidade com o pacote 6
import os, time, shutil, zipfile, requests
import pandas as pd

LINK = os.environ["PADCHEST_LINK"]   # link privado da BIMCV, fora do repositório (cláusula 6)

host, token = LINK.rstrip("/").split("/s/")
token = token.split("/")[0]

def candidatos(nome):
    return [f"{host}/public.php/dav/files/{token}/{nome}",
            f"{host}/s/{token}/download?path=%2F&files={nome}",
            f"{host}/index.php/s/{token}/download?path=%2F&files={nome}"]

def achar_url(nome):
    for u in candidatos(nome):
        try:
            r = requests.get(u, headers={"Range": "bytes=0-1023"}, stream=True, timeout=60)
            ok = r.status_code in (200, 206) and "html" not in r.headers.get("Content-Type", "")
            r.close()
            if ok:
                return u
        except Exception:
            pass
    return None

def baixar(url, destino, tentativas=10):
    t0 = time.time()
    for k in range(tentativas):
        ja = os.path.getsize(destino) if os.path.exists(destino) else 0
        try:
            with requests.get(url, headers={"Range": f"bytes={ja}-"} if ja else {}, stream=True, timeout=120) as r:
                if r.status_code == 416:
                    break                                  # já estava completo
                r.raise_for_status()
                with open(destino, "ab" if (ja and r.status_code == 206) else "wb") as f:
                    for b in r.iter_content(16 * 1024 * 1024):
                        f.write(b)
            break
        except Exception as e:
            print(f"  tentativa {k + 1} caiu ({type(e).__name__}); retomando...")
            time.sleep(10)
    return time.time() - t0

livre = shutil.disk_usage("/scratch").free / 1e9
print(f"Espaço livre no /scratch: {livre:.0f} GB")

url = achar_url("6.zip")
print("Endereço de download encontrado:", "sim" if url else "NÃO")

os.makedirs("/scratch/padchest", exist_ok=True)
destino = "/scratch/padchest/6.zip"
dt = baixar(url, destino)
gb = os.path.getsize(destino) / 1e9
print(f"Pacote 6: {gb:.1f} GB em {dt / 60:.1f} min | {gb * 1000 / dt:.0f} MB/s")
print(f"Estimativa para os 282 GB do recorte: {282 / (gb / dt) / 3600:.1f} h")

rec = pd.read_csv("/out/padchest_recorte.csv")
alvo = set(rec.loc[rec["ImageDir"] == 6, "ImageID"])
with zipfile.ZipFile(destino) as z:
    nomes = {os.path.basename(n): n for n in z.namelist() if n.lower().endswith(".png")}
print(f"PNG no zip: {len(nomes)} (planilha da BIMCV: 1.645)")
print(f"Imagens do recorte no pacote 6: {len(alvo)} | encontradas no zip: {len(alvo & set(nomes))} | faltando: {len(alvo - set(nomes))}")

In [ ]:
# Bloco P2b — pipeline por pacote: baixar (4 partes) → conferir → guardar em /out → apagar o zip
import os, re, time, zipfile, hashlib, shutil, requests
import numpy as np, pandas as pd, cv2
from concurrent.futures import ThreadPoolExecutor

LINK = os.environ["PADCHEST_LINK"]   # link privado da BIMCV, fora do repositório (cláusula 6)
host, token = LINK.rstrip("/").split("/s/")
token = token.split("/")[0]

SAIDA, TMP = "/out/padchest", "/scratch/padchest"
for p in [SAIDA, f"{SAIDA}/insp100", f"{SAIDA}/c320", TMP]:
    os.makedirs(p, exist_ok=True)

rec = pd.read_csv("/out/padchest_recorte.csv")

# amostra fixa da inspeção da máscara: 50 AP (AP e AP deitado) + 50 PA, semente 2026
arq_amostra = f"{SAIDA}/amostra_insp100.csv"
if not os.path.exists(arq_amostra):
    ap = rec[rec["Projection"].isin(["AP", "AP_horizontal"])].sample(50, random_state=2026)
    pa = rec[rec["Projection"] == "PA"].sample(50, random_state=2026)
    pd.concat([ap, pa]).assign(n=range(1, 101)).to_csv(arq_amostra, index=False)
ids_insp = set(pd.read_csv(arq_amostra)["ImageID"])

def achar_url(nome):
    for u in [f"{host}/public.php/dav/files/{token}/{nome}",
              f"{host}/s/{token}/download?path=%2F&files={nome}",
              f"{host}/index.php/s/{token}/download?path=%2F&files={nome}"]:
        try:
            r = requests.get(u, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
            st, cr, ct = r.status_code, r.headers.get("Content-Range"), r.headers.get("Content-Type", "")
            tam = r.headers.get("Content-Length")
            r.close()
            if "html" in ct:
                continue
            if st == 206 and cr:
                return u, int(cr.split("/")[-1]), True
            if st == 200 and tam:
                return u, int(tam), False
        except Exception:
            pass
    raise IOError(f"não achei o endereço de {nome}")

def baixar(url, total, destino, em_partes, partes=4):
    with open(destino, "wb") as f:
        f.truncate(total)
    n = partes if em_partes else 1
    limites = [(i * total // n, (i + 1) * total // n - 1) for i in range(n)]
    def parte(i):
        a, b = limites[i]
        pos = a
        for _ in range(30):
            if pos > b:
                return
            try:
                h = {"Range": f"bytes={pos}-{b}"} if em_partes else {}
                with requests.get(url, headers=h, stream=True, timeout=120) as r:
                    r.raise_for_status()
                    with open(destino, "r+b") as f:
                        f.seek(pos)
                        for blk in r.iter_content(8 * 1024 * 1024):
                            f.write(blk)
                            pos += len(blk)
            except Exception:
                if not em_partes:
                    pos = a
                time.sleep(10)
        if pos <= b:
            raise IOError(f"parte {i} incompleta")
    with ThreadPoolExecutor(n) as ex:
        list(ex.map(parte, range(n)))

def numero(v):
    m = re.search(r"-?\d+(\.\d+)?", str(v))
    return float(m.group()) if m else np.nan

def processar(n):
    csv_saida = f"{SAIDA}/pixels_pacote_{n}.csv"
    if os.path.exists(csv_saida):
        print(f"Pacote {n}: já processado, pulando")
        return
    alvo = rec[rec["ImageDir"] == n]
    zipf = f"{TMP}/{n}.zip"
    url, total, em_partes = achar_url(f"{n}.zip")
    print(f"Pacote {n}: {total / 1e9:.1f} GB | download {'em 4 partes' if em_partes else 'em 1 parte'} | "
          f"espaço livre {shutil.disk_usage(TMP).free / 1e9:.0f} GB")
    t0 = time.time()
    baixar(url, total, zipf, em_partes)
    t_down = time.time() - t0
    print(f"  baixado em {t_down / 60:.1f} min | {total / 1e6 / t_down:.1f} MB/s")

    regs = []
    t1 = time.time()
    with zipfile.ZipFile(zipf) as z:
        nomes = {os.path.basename(m): m for m in z.namelist() if m.lower().endswith(".png")}
        n_png = len(nomes)
        for row in alvo.itertuples():
            reg = {"ImageID": row.ImageID, "no_zip": row.ImageID in nomes}
            try:
                b = z.read(nomes[row.ImageID])
                reg["md5"] = hashlib.md5(b).hexdigest()
                img = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_UNCHANGED)
                if img is None:
                    raise ValueError("não decodificou")
                g = img if img.ndim == 2 else img[..., 0]
                amostra = g[::4, ::4]
                p = np.percentile(amostra, [0.5, 50, 99.5])
                wc, ww = numero(row.WindowCenter_DICOM), numero(row.WindowWidth_DICOM)
                dentro = np.mean((amostra >= wc - ww / 2) & (amostra <= wc + ww / 2)) if ww == ww else np.nan
                reg.update(abriu=True, dtype=str(img.dtype), canais=1 if img.ndim == 2 else img.shape[2],
                           alt=g.shape[0], larg=g.shape[1], vmin=int(g.min()), vmax=int(g.max()),
                           p005=p[0], p50=p[1], p995=p[2], frac_janela=dentro)
                cv2.imwrite(f"{SAIDA}/c320/{row.ImageID}", cv2.resize(g, (320, 320), interpolation=cv2.INTER_AREA))
                if row.ImageID in ids_insp:
                    lo, hi = np.percentile(g, [0.5, 99.5])
                    img8 = (np.clip((g.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1) * 255).round().astype(np.uint8)
                    cv2.imwrite(f"{SAIDA}/insp100/{row.ImageID}", img8)
            except Exception as e:
                reg.update(abriu=False, erro=str(e)[:80])
            regs.append(reg)

    px = alvo.merge(pd.DataFrame(regs), on="ImageID")
    px["pacote_png"], px["t_download_min"], px["mb_s"] = n_png, t_down / 60, total / 1e6 / t_down
    px.to_csv(csv_saida, index=False)
    os.remove(zipf)
    print(f"  {len(alvo)} imagens do recorte conferidas em {(time.time() - t1) / 60:.1f} min | "
          f"abriram {int(px['abriu'].sum())} | PNG no zip {n_png} | zip apagado")

processar(6)

In [ ]:
# Bloco P2c — resumo dos pixels do pacote 6 e miniaturas (lê só o /out)
import os, glob
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt

px = pd.read_csv("/out/padchest/pixels_pacote_6.csv")
ok = px[px["abriu"]]
px["bits_csv"] = pd.to_numeric(px["BitsStored_DICOM"], errors="coerce")
ok = px[px["abriu"]].copy()
ok["cabe_nos_bits"] = ok["vmax"] <= 2 ** ok["bits_csv"] - 1
ok["dim_bate"] = (ok["alt"] == pd.to_numeric(ok["Rows_DICOM"], errors="coerce")) & \
                 (ok["larg"] == pd.to_numeric(ok["Columns_DICOM"], errors="coerce"))

print("Abriu:", px["abriu"].value_counts().to_dict())
print("Formato (tipo, canais):", ok.groupby(["dtype", "canais"]).size().to_dict())
print("Dimensões batem com o CSV:", round(ok["dim_bate"].mean(), 4),
      "| maior valor cabe nos bits do CSV:", round(ok["cabe_nos_bits"].mean(), 4))

print("\nPor fabricante:")
print(ok.groupby("Manufacturer_DICOM").agg(
    imagens=("ImageID", "size"), positivos=("pos", "sum"),
    bits_csv=("bits_csv", lambda s: sorted(s.dropna().unique().astype(int).tolist())),
    fotometria=("PhotometricInterpretation_DICOM", lambda s: s.value_counts().to_dict()),
    vmin_mediana=("vmin", "median"), vmax_mediana=("vmax", "median"),
    p005_mediana=("p005", "median"), p995_mediana=("p995", "median"),
    frac_janela_mediana=("frac_janela", "median"),
    alt_mediana=("alt", "median"), larg_mediana=("larg", "median")).T.to_string())

dup = ok[ok.duplicated("md5", keep=False)]
print(f"\nDuplicatas exatas no pacote 6: {len(dup)} arquivos em {dup['md5'].nunique()} grupos")

tam = sum(os.path.getsize(f) for f in glob.glob("/out/padchest/c320/*.png")) / 1e9
print(f"Cópia de 320 px: {tam:.2f} GB para {len(ok)} imagens → estimativa total {tam * 24773 / len(ok):.1f} GB")
print("Imagens da amostra de inspeção já guardadas:", len(glob.glob("/out/padchest/insp100/*.png")), "de 100")

mini = pd.concat([g.head(4) for _, g in ok.groupby("Manufacturer_DICOM")])
fig, axes = plt.subplots(1, len(mini), figsize=(4 * len(mini), 4))
for ax, row in zip(np.atleast_1d(axes), mini.itertuples()):
    im = cv2.imread(f"/out/padchest/c320/{row.ImageID}", cv2.IMREAD_UNCHANGED).astype(np.float32)
    lo, hi = np.percentile(im, [0.5, 99.5])
    ax.imshow(np.clip((im - lo) / max(hi - lo, 1), 0, 1), cmap="gray")
    ax.set_title(f"{str(row.Manufacturer_DICOM)[:18]}\n{row.PhotometricInterpretation_DICOM}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Bloco P3 — todos os pacotes do recorte: baixar (8 partes) → conferir → guardar em /out → apagar o zip
import os, re, time, zipfile, hashlib, shutil, requests
import numpy as np, pandas as pd, cv2
from concurrent.futures import ThreadPoolExecutor

LINK = os.environ["PADCHEST_LINK"]   # link privado da BIMCV, fora do repositório (cláusula 6)
host, token = LINK.rstrip("/").split("/s/")
token = token.split("/")[0]

SAIDA, TMP = "/out/padchest", "/scratch/padchest"
for p in [SAIDA, f"{SAIDA}/insp100", f"{SAIDA}/c320", TMP]:
    os.makedirs(p, exist_ok=True)

rec = pd.read_csv("/out/padchest_recorte.csv")
arq_amostra = f"{SAIDA}/amostra_insp100.csv"
if not os.path.exists(arq_amostra):
    ap = rec[rec["Projection"].isin(["AP", "AP_horizontal"])].sample(50, random_state=2026)
    pa = rec[rec["Projection"] == "PA"].sample(50, random_state=2026)
    pd.concat([ap, pa]).assign(n=range(1, 101)).to_csv(arq_amostra, index=False)
ids_insp = set(pd.read_csv(arq_amostra)["ImageID"])

# refazer o pacote 6 se ele ainda não tem as medidas novas
p6 = f"{SAIDA}/pixels_pacote_6.csv"
if os.path.exists(p6) and "frac_mult16" not in pd.read_csv(p6, nrows=1).columns:
    os.remove(p6)

def achar_url(nome):
    for u in [f"{host}/public.php/dav/files/{token}/{nome}",
              f"{host}/s/{token}/download?path=%2F&files={nome}",
              f"{host}/index.php/s/{token}/download?path=%2F&files={nome}"]:
        try:
            r = requests.get(u, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
            st, cr, ct = r.status_code, r.headers.get("Content-Range"), r.headers.get("Content-Type", "")
            tam = r.headers.get("Content-Length")
            r.close()
            if "html" in ct:
                continue
            if st == 206 and cr:
                return u, int(cr.split("/")[-1]), True
            if st == 200 and tam:
                return u, int(tam), False
        except Exception:
            pass
    raise IOError(f"não achei o endereço de {nome}")

def baixar(url, total, destino, em_partes, partes=8):
    with open(destino, "wb") as f:
        f.truncate(total)
    n = partes if em_partes else 1
    limites = [(i * total // n, (i + 1) * total // n - 1) for i in range(n)]
    def parte(i):
        a, b = limites[i]
        pos = a
        for _ in range(30):
            if pos > b:
                return
            try:
                h = {"Range": f"bytes={pos}-{b}"} if em_partes else {}
                with requests.get(url, headers=h, stream=True, timeout=120) as r:
                    r.raise_for_status()
                    with open(destino, "r+b") as f:
                        f.seek(pos)
                        for blk in r.iter_content(8 * 1024 * 1024):
                            f.write(blk)
                            pos += len(blk)
            except Exception:
                if not em_partes:
                    pos = a
                time.sleep(10)
        if pos <= b:
            raise IOError(f"parte {i} incompleta")
    with ThreadPoolExecutor(n) as ex:
        list(ex.map(parte, range(n)))

def numero(v):
    m = re.search(r"-?\d+(\.\d+)?", str(v))
    return float(m.group()) if m else np.nan

def processar(n):
    csv_saida = f"{SAIDA}/pixels_pacote_{n}.csv"
    if os.path.exists(csv_saida):
        print(f"Pacote {n}: já processado, pulando")
        return
    alvo = rec[rec["ImageDir"] == n]
    zipf = f"{TMP}/{n}.zip"
    url, total, em_partes = achar_url(f"{n}.zip")
    print(f"Pacote {n}: {total / 1e9:.1f} GB | {len(alvo)} imagens do recorte | espaço livre {shutil.disk_usage(TMP).free / 1e9:.0f} GB")
    t0 = time.time()
    baixar(url, total, zipf, em_partes)
    t_down = time.time() - t0
    print(f"  baixado em {t_down / 60:.1f} min | {total / 1e6 / t_down:.1f} MB/s")

    regs = []
    t1 = time.time()
    with zipfile.ZipFile(zipf) as z:
        nomes = {os.path.basename(m): m for m in z.namelist() if m.lower().endswith(".png")}
        n_png = len(nomes)
        for row in alvo.itertuples():
            reg = {"ImageID": row.ImageID, "no_zip": row.ImageID in nomes}
            try:
                b = z.read(nomes[row.ImageID])
                reg["md5"] = hashlib.md5(b).hexdigest()
                img = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_UNCHANGED)
                if img is None:
                    raise ValueError("não decodificou")
                g = img if img.ndim == 2 else img[..., 0]
                a = g[::4, ::4]
                v = a.ravel()
                p = np.percentile(v, [0.5, 50, 99.5])
                wc, ww = numero(row.WindowCenter_DICOM), numero(row.WindowWidth_DICOM)
                if ww == ww:
                    dentro = np.mean((v >= wc - ww / 2) & (v <= wc + ww / 2))
                    dentro16 = np.mean((v >= 16 * (wc - ww / 2)) & (v <= 16 * (wc + ww / 2)))
                else:
                    dentro = dentro16 = np.nan
                reg.update(abriu=True, dtype=str(img.dtype), canais=1 if img.ndim == 2 else img.shape[2],
                           alt=g.shape[0], larg=g.shape[1], vmin=int(g.min()), vmax=int(g.max()),
                           p005=p[0], p50=p[1], p995=p[2], frac_janela=dentro, frac_janela_x16=dentro16,
                           frac_mult16=np.mean(v % 16 == 0), frac_mult64=np.mean(v % 64 == 0),
                           frac_zero=np.mean(v == 0), frac_teto=np.mean(v >= 65519),
                           n_valores=int(np.unique(v).size))
                cv2.imwrite(f"{SAIDA}/c320/{row.ImageID}", cv2.resize(g, (320, 320), interpolation=cv2.INTER_AREA))
                if row.ImageID in ids_insp:
                    lo, hi = np.percentile(g, [0.5, 99.5])
                    img8 = (np.clip((g.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1) * 255).round().astype(np.uint8)
                    cv2.imwrite(f"{SAIDA}/insp100/{row.ImageID}", img8)
            except Exception as e:
                reg.update(abriu=False, erro=str(e)[:80])
            regs.append(reg)

    px = alvo.merge(pd.DataFrame(regs), on="ImageID")
    px["pacote_png"], px["t_download_min"], px["mb_s"] = n_png, t_down / 60, total / 1e6 / t_down
    px.to_csv(csv_saida, index=False)
    os.remove(zipf)
    print(f"  {len(alvo)} conferidas em {(time.time() - t1) / 60:.1f} min | abriram {int(px['abriu'].sum())} | "
          f"PNG no zip {n_png} | zip apagado")

inicio = time.time()
for n in [6] + [k for k in list(range(12)) + [54] if k != 6]:
    try:
        processar(n)
    except Exception as e:
        print(f"Pacote {n}: FALHOU ({type(e).__name__}: {str(e)[:100]}); seguindo para o próximo")
        if os.path.exists(f"{TMP}/{n}.zip"):
            os.remove(f"{TMP}/{n}.zip")
print(f"\nFim: {(time.time() - inicio) / 3600:.1f} h | pacotes prontos: "
      f"{sorted(int(f.split('_')[-1][:-4]) for f in os.listdir(SAIDA) if f.startswith('pixels_pacote_'))}")

In [ ]:
import os
# Bloco D1 — diagnóstico do servidor da BIMCV (1 conexão, sem baixar o pacote)
import time, requests

LINK = os.environ["PADCHEST_LINK"]   # link privado da BIMCV, fora do repositório (cláusula 6)
host, token = LINK.rstrip("/").split("/s/")
url = f"{host}/public.php/dav/files/{token}/0.zip"

for nome, faixa in [("início", "bytes=0-1023"), ("meio", "bytes=10000000000-10000001023")]:
    try:
        r = requests.get(url, headers={"Range": faixa}, stream=True, timeout=60)
        print(f"{nome}: status {r.status_code} | Content-Range {r.headers.get('Content-Range')} | "
              f"Retry-After {r.headers.get('Retry-After')} | tipo {r.headers.get('Content-Type')}")
        if r.status_code >= 400:
            print("   resposta:", r.text[:300].replace("\n", " "))
        r.close()
    except Exception as e:
        print(f"{nome}: ERRO {type(e).__name__}: {str(e)[:200]}")

# velocidade de 1 conexão em 200 MB
t0 = time.time(); n = 0
try:
    with requests.get(url, headers={"Range": "bytes=0-199999999"}, stream=True, timeout=120) as r:
        print("teste de velocidade: status", r.status_code)
        for b in r.iter_content(8 * 1024 * 1024):
            n += len(b)
    print(f"1 conexão: {n / 1e6:.0f} MB em {time.time() - t0:.0f} s | {n / 1e6 / (time.time() - t0):.1f} MB/s")
except Exception as e:
    print(f"teste de velocidade: ERRO {type(e).__name__}: {str(e)[:200]} | baixou {n / 1e6:.0f} MB")

In [ ]:
# Bloco P3b — todos os pacotes do recorte, versão robusta (4 partes, confere 206, espera crescente)
import os, re, time, zipfile, hashlib, shutil, requests
import numpy as np, pandas as pd, cv2
from concurrent.futures import ThreadPoolExecutor

LINK = os.environ["PADCHEST_LINK"]   # link privado da BIMCV, fora do repositório (cláusula 6)
host, token = LINK.rstrip("/").split("/s/")
token = token.split("/")[0]

SAIDA, TMP = "/out/padchest", "/scratch/padchest"
for p in [SAIDA, f"{SAIDA}/insp100", f"{SAIDA}/c320", TMP]:
    os.makedirs(p, exist_ok=True)
for f in os.listdir(TMP):                      # restos de downloads interrompidos
    os.remove(os.path.join(TMP, f))

rec = pd.read_csv("/out/padchest_recorte.csv")
ids_insp = set(pd.read_csv(f"{SAIDA}/amostra_insp100.csv")["ImageID"])

def url_e_tamanho(nome):
    u = f"{host}/public.php/dav/files/{token}/{nome}"
    r = requests.get(u, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
    st, cr = r.status_code, r.headers.get("Content-Range")
    r.close()
    if st != 206 or not cr:
        raise IOError(f"{nome}: status {st}")
    return u, int(cr.split("/")[-1])

def baixar(url, total, destino, partes=4):
    with open(destino, "wb") as f:
        f.truncate(total)
    limites = [(i * total // partes, (i + 1) * total // partes - 1) for i in range(partes)]
    def parte(i):
        a, b = limites[i]
        pos, ultimo = a, ""
        for k in range(60):
            if pos > b:
                return
            try:
                with requests.get(url, headers={"Range": f"bytes={pos}-{b}"}, stream=True, timeout=(30, 120)) as r:
                    if r.status_code != 206:
                        raise IOError(f"status {r.status_code}")
                    with open(destino, "r+b") as f:
                        f.seek(pos)
                        for blk in r.iter_content(8 * 1024 * 1024):
                            f.write(blk)
                            pos += len(blk)
            except Exception as e:
                ultimo = f"{type(e).__name__}: {str(e)[:80]}"
                time.sleep(min(15 * (k + 1), 120))
        if pos <= b:
            raise IOError(f"parte {i} incompleta após 60 tentativas; último erro: {ultimo}")
    with ThreadPoolExecutor(partes) as ex:
        list(ex.map(parte, range(partes)))

def numero(v):
    m = re.search(r"-?\d+(\.\d+)?", str(v))
    return float(m.group()) if m else np.nan

def processar(n):
    csv_saida = f"{SAIDA}/pixels_pacote_{n}.csv"
    if os.path.exists(csv_saida):
        print(f"Pacote {n}: já processado, pulando")
        return
    alvo = rec[rec["ImageDir"] == n]
    zipf = f"{TMP}/{n}.zip"
    url, total = url_e_tamanho(f"{n}.zip")
    print(f"Pacote {n}: {total / 1e9:.1f} GB | {len(alvo)} imagens do recorte")
    t0 = time.time()
    try:
        baixar(url, total, zipf)
    except Exception:
        if os.path.exists(zipf):
            os.remove(zipf)
        raise
    t_down = time.time() - t0
    print(f"  baixado em {t_down / 60:.1f} min | {total / 1e6 / t_down:.1f} MB/s")

    regs = []
    t1 = time.time()
    with zipfile.ZipFile(zipf) as z:
        nomes = {os.path.basename(m): m for m in z.namelist() if m.lower().endswith(".png")}
        n_png = len(nomes)
        for row in alvo.itertuples():
            reg = {"ImageID": row.ImageID, "no_zip": row.ImageID in nomes}
            try:
                b = z.read(nomes[row.ImageID])
                reg["md5"] = hashlib.md5(b).hexdigest()
                img = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_UNCHANGED)
                if img is None:
                    raise ValueError("não decodificou")
                g = img if img.ndim == 2 else img[..., 0]
                v = g[::4, ::4].ravel()
                p = np.percentile(v, [0.5, 50, 99.5])
                wc, ww = numero(row.WindowCenter_DICOM), numero(row.WindowWidth_DICOM)
                if ww == ww:
                    dentro = np.mean((v >= wc - ww / 2) & (v <= wc + ww / 2))
                    dentro16 = np.mean((v >= 16 * (wc - ww / 2)) & (v <= 16 * (wc + ww / 2)))
                else:
                    dentro = dentro16 = np.nan
                reg.update(abriu=True, dtype=str(img.dtype), canais=1 if img.ndim == 2 else img.shape[2],
                           alt=g.shape[0], larg=g.shape[1], vmin=int(g.min()), vmax=int(g.max()),
                           p005=p[0], p50=p[1], p995=p[2], frac_janela=dentro, frac_janela_x16=dentro16,
                           frac_mult16=np.mean(v % 16 == 0), frac_mult64=np.mean(v % 64 == 0),
                           frac_zero=np.mean(v == 0), frac_teto=np.mean(v >= 65519),
                           n_valores=int(np.unique(v).size))
                cv2.imwrite(f"{SAIDA}/c320/{row.ImageID}", cv2.resize(g, (320, 320), interpolation=cv2.INTER_AREA))
                if row.ImageID in ids_insp:
                    lo, hi = np.percentile(g, [0.5, 99.5])
                    img8 = (np.clip((g.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1) * 255).round().astype(np.uint8)
                    cv2.imwrite(f"{SAIDA}/insp100/{row.ImageID}", img8)
            except Exception as e:
                reg.update(abriu=False, erro=str(e)[:80])
            regs.append(reg)

    px = alvo.merge(pd.DataFrame(regs), on="ImageID")
    px["pacote_png"], px["t_download_min"], px["mb_s"] = n_png, t_down / 60, total / 1e6 / t_down
    px.to_csv(csv_saida, index=False)
    os.remove(zipf)
    print(f"  {len(alvo)} conferidas em {(time.time() - t1) / 60:.1f} min | abriram {int(px['abriu'].sum())} | "
          f"PNG no zip {n_png} | zip apagado")

pacotes = list(range(12)) + [54]
inicio = time.time()
for rodada in [1, 2]:
    falhas = []
    for n in pacotes:
        try:
            processar(n)
        except Exception as e:
            print(f"Pacote {n}: FALHOU ({type(e).__name__}: {str(e)[:150]})")
            falhas.append(n)
    if not falhas:
        break
    print(f"\nRodada {rodada} terminou com falhas em {falhas}; pausa de 2 min antes de tentar de novo\n")
    pacotes = falhas
    time.sleep(120)

prontos = sorted(int(f.split("_")[-1][:-4]) for f in os.listdir(SAIDA) if f.startswith("pixels_pacote_"))
print(f"\nFim: {(time.time() - inicio) / 3600:.1f} h | pacotes prontos ({len(prontos)} de 13): {prontos}")

In [ ]:
# Bloco P4 — resumo dos 13 pacotes do PadChest (lê só o /out)
import os, glob
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt

SAIDA = "/out/padchest"
arqs = sorted(glob.glob(f"{SAIDA}/pixels_pacote_*.csv"))
px = pd.concat([pd.read_csv(a) for a in arqs], ignore_index=True)
print(f"Pacotes lidos: {len(arqs)} de 13 | imagens do recorte: {len(px)}")

# 1. integridade
planilha = {0: 3004, 1: 3024, 2: 2984, 3: 2992, 4: 2993, 5: 2947, 6: 1645, 7: 2742,
            8: 2991, 9: 2990, 10: 2975, 11: 3022, 54: 4297}
integ = px.groupby("ImageDir").agg(png_no_zip=("pacote_png", "first"), recorte=("ImageID", "size"),
                                   abriram=("abriu", "sum"), mb_s=("mb_s", "first"))
integ["planilha"] = integ.index.map(planilha)
integ["bate"] = integ["png_no_zip"] == integ["planilha"]
print("\n=== Integridade por pacote")
print(integ.to_string())
print("Não abriram:", int((~px["abriu"].astype(bool)).sum()))

ok = px[px["abriu"].astype(bool)].copy()
ok["bits_csv"] = pd.to_numeric(ok["BitsStored_DICOM"], errors="coerce")
ok["fab"] = ok["Manufacturer_DICOM"].astype(str).str.replace("CompanyLtd", "").str.replace("MedicalSystems", "")
print("\nFormato:", ok.groupby(["dtype", "canais"]).size().to_dict())
dim = (ok["alt"] == pd.to_numeric(ok["Rows_DICOM"], errors="coerce")) & \
      (ok["larg"] == pd.to_numeric(ok["Columns_DICOM"], errors="coerce"))
print("Dimensões batem com o CSV:", round(dim.mean(), 4))

# 2. escala dos valores, janela e saturação
print("\n=== Escala, janela e saturação (medianas) por fabricante e bits do CSV")
print(ok.groupby(["fab", "bits_csv"]).agg(
    imagens=("ImageID", "size"),
    frac_mult16=("frac_mult16", "median"), frac_mult64=("frac_mult64", "median"),
    n_valores=("n_valores", "median"), vmax=("vmax", "median"),
    janela_original=("frac_janela", "median"), janela_x16=("frac_janela_x16", "median"),
    frac_teto=("frac_teto", "median"), frac_zero=("frac_zero", "median"),
    pct_imagens_com_1pct_no_teto=("frac_teto", lambda s: round((s > 0.01).mean(), 3))).round(4).to_string())

# 3. duplicatas exatas (mesma regra do Plus e do RSNA)
dup = ok[ok.duplicated("md5", keep=False)]
grp = dup.groupby("md5")["PatientID"].nunique()
sai_entre = ok["md5"].isin(grp[grp > 1].index)
mesmo = ok[ok["md5"].isin(grp[grp == 1].index)].sort_values(["md5", "ImageID"])
sai_mesmo = ok.index.isin(mesmo[mesmo.duplicated("md5", keep="first")].index)
print(f"\nDuplicatas exatas: {len(dup)} arquivos em {dup['md5'].nunique()} grupos "
      f"({int((grp > 1).sum())} grupos entre pacientes diferentes)")
final = ok[~sai_entre & ~sai_mesmo].copy()

# 4. recontagem final
print(f"\n=== Base final do PadChest: {len(final)} estudos | {final['PatientID'].nunique()} pacientes | "
      f"positivos {int(final['pos'].sum())} | com atípica {int(final['pos_atipica'].sum())} | "
      f"prevalência {final['pos'].mean():.2%} | meta 750")

# 5. composição
final["faixa_idade"] = pd.cut(final["idade"], [18, 40, 60, 75, 101], right=False,
                              labels=["18–39", "40–59", "60–74", "75+"])
for c in ["Projection", "fab", "PatientSex_DICOM", "faixa_idade"]:
    g = final.groupby(c, observed=True)["pos"]
    print(f"\n=== {c}")
    print(pd.DataFrame({"imagens": g.size(), "positivos": g.sum(), "prevalencia": g.mean().round(4)}).to_string())

final.to_csv(f"{SAIDA}/padchest_final.csv", index=False)

# 6. espaço
c320 = sum(os.path.getsize(f) for f in glob.glob(f"{SAIDA}/c320/*.png")) / 1e9
insp = glob.glob(f"{SAIDA}/insp100/*.png")
print(f"\nCópia de 320 px: {c320:.2f} GB | amostra de inspeção: {len(insp)} de 100 "
      f"({sum(os.path.getsize(f) for f in insp) / 1e9:.2f} GB)")

# 7. miniaturas para conferir inversão
mini = pd.concat([g.sample(4, random_state=1) for _, g in final.groupby("fab")])
fig, axes = plt.subplots(1, len(mini), figsize=(3 * len(mini), 3.5))
for ax, row in zip(axes, mini.itertuples()):
    im = cv2.imread(f"{SAIDA}/c320/{row.ImageID}", cv2.IMREAD_UNCHANGED).astype(np.float32)
    lo, hi = np.percentile(im, [0.5, 99.5])
    ax.imshow(np.clip((im - lo) / max(hi - lo, 1), 0, 1), cmap="gray")
    ax.set_title(f"{row.fab}\n{row.Projection}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Bloco P5 — máscara do pulmão no PadChest: inspeção às cegas de 100 imagens (mesmo procedimento das outras bases)
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from lungs_segmentation.pre_trained_models import create_model

SAIDA = "/out/padchest"
insp = pd.read_csv(f"{SAIDA}/amostra_insp100.csv").sort_values("n").reset_index(drop=True)
model = create_model("resnet34").eval()

def maior_componente(m):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    if n <= 1:
        return m.astype(np.uint8)
    return (lab == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)

def segmentar_roi(img8, limiar=0.2):
    x = np.repeat(img8[..., None], 3, axis=2).astype(np.float32)
    x = (x - x.min()) / max(x.max() - x.min(), 1e-6)
    x = cv2.resize(x, (512, 512), interpolation=cv2.INTER_LINEAR)
    with torch.no_grad():
        p = torch.sigmoid(model(torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)))[0].numpy()
    pulmoes = np.stack([maior_componente(m) for m in (p > limiar)])
    return x[..., 0], pulmoes.max(axis=0), pulmoes

# as imagens já estão em 8 bits por percentis 0,5–99,5 (todas MONOCHROME2, sem inversão)
res = [segmentar_roi(cv2.imread(f"{SAIDA}/insp100/{i}", cv2.IMREAD_GRAYSCALE)) for i in insp["ImageID"]]
insp["area_roi"] = [r[1].mean() for r in res]
insp["area_p1"] = [r[2][0].mean() for r in res]
insp["area_p2"] = [r[2][1].mean() for r in res]
insp["razao_pulmoes"] = insp[["area_p1", "area_p2"]].min(axis=1) / insp[["area_p1", "area_p2"]].max(axis=1)
insp.to_csv(f"{SAIDA}/roi_inspecao_100_padchest.csv", index=False)

for pg in range(4):
    fig, axes = plt.subplots(5, 5, figsize=(20, 20))
    for k, ax in enumerate(axes.flat):
        i = pg * 25 + k
        ax.imshow(res[i][0], cmap="gray")
        ax.contour(res[i][1], levels=[0.5], colors="r", linewidths=1)
        proj = "AP" if insp.loc[i, "Projection"] != "PA" else "PA"
        ax.set_title(f"#{i + 1} {proj}", fontsize=14)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"{SAIDA}/roi_inspecao_padchest_{pg + 1}.png", dpi=60)
    plt.show()

print("Razão entre pulmões < 0,35 (alerta):", insp.loc[insp["razao_pulmoes"] < 0.35, "n"].tolist())
print(insp.groupby(insp["Projection"].ne("PA").map({True: "AP", False: "PA"}))["area_roi"].describe().round(3).to_string())

In [ ]:
# Bloco P6 — ampliar casos da inspeção do PadChest (às cegas)
casos = [5, 8, 10, 15, 40, 43, 93]
for parte, bloco in enumerate((casos[:4], casos[4:]), start=1):
    fig, axes = plt.subplots(len(bloco), 2, figsize=(14, 7 * len(bloco)))
    for (a1, a2), n in zip(axes, bloco):
        i = n - 1
        img = cv2.imread(f"{SAIDA}/insp100/{insp.loc[i, 'ImageID']}", cv2.IMREAD_GRAYSCALE)
        roi_full = cv2.resize(res[i][1], (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        proj = "AP" if insp.loc[i, "Projection"] != "PA" else "PA"
        a1.imshow(img, cmap="gray")
        a1.set_title(f"#{n} {proj} — sem máscara", fontsize=13)
        a1.axis("off")
        a2.imshow(img, cmap="gray")
        a2.contour(roi_full, levels=[0.5], colors="r", linewidths=1.5)
        a2.set_title(f"#{n} — área {insp.loc[i, 'area_roi']:.2f} · razão entre pulmões {insp.loc[i, 'razao_pulmoes']:.2f}",
                     fontsize=13)
        a2.axis("off")
    plt.tight_layout()
    plt.savefig(f"{SAIDA}/roi_casos_ampliados_{parte}.png", dpi=60)
    plt.show()